# PROJET DE TRAITEMENTS DISTRIBUES
Edoardo PICIUCCHI, Aurélien DUVIGNAC-ROSA, Jean-Marc FAUVEL

# Présentation

Le projet consiste en l'étude d'un article intitulé : « CCF : Fast and Scalable Connected Component Computation in MapReduce » (« CCF : Calcul de composantes connexes rapide et scalable en MapReduce »). Cet article a été publié en 2013 par Jimmy Lin et Michael Schatz. Il propose un algorithme de calcul de composantes connexes en utilisant le modèle de programmation MapReduce. Cet algorithme est basé sur l'algorithme de Label Propagation. Il est conçu pour être rapide et scalable, c'est-à-dire qu'il peut être utilisé sur de grands ensembles de données et sur un grand nombre de machines.

L'algorithme a pour objectif d’identifier les sous-graphes connectés au sein d’un graphe G. Le graphe analysé doit être représenté par une collection de paires (N1, N2) où N1 et N2 sont des noeuds du graph G qui ont une connexion entre eux. Le couple (N1, N2) représente une arête du graphe.
L’algorithme présenté dans ce papier va permettre d’identifier des sous-graphes connectés entre eux en modifiant les paires des sous-graphes afin qu’elles soient toutes constituées ainsi : (N, Nmin) où N est un noeud du sous graph et Nmin le noeud de plus petite valeur appartenant au même sous-graphe.
Par ailleurs, l’algorithme ayant pour objet de permettre de traiter des graphes de grande taille, il adopte une programmation distribuée de type MapReduce, permettant ainsi de répartir le traitement sur plusieurs machines et permettre le traitement des données de manière parallèle.

## Objectifs du projet

1. Lire, comprendre et expliquer l’algorithme décrit dans le papier de Hakan Kardes, Siddharth Agrawal, Xin Wang et Ang Sun intitulé « CCF : Fast and Scalable Connected Component Computation in MapReduce » ;
2. Coder l’algorithme en Spark en utilisant à la fois des RDD et des DataFrames ;
3. L’implémentation doit être exécutée en Python, et optionnellement en Scala
4. Effectuer une analyse comparative des versions en RDD et DataFrame sur des graphes de tailles croissantes ;
5. Utilisation de DataBricks pour les petits graphes, et Google Cloud Cluster pour les plus gros (<20 Gb).

La première étape a consisté à implémenter l'algorithme CCF en se limitant exclusivement à l'utilisation de python sans le paradigme spark. Cela nous a permis de mieux appréhender l'algorithme et d'en comprendre les différentes étapes. Cette étape réalisée, nous avons pu utiliser les éléments relatifs à la bibiliothèque `spark`.
L'implémentation décrite ci-dessous permet de réaliser les étapes de l'algorithme CCF décrites dans l'exemple de l'article.
Les résultats obtenus nous ont permis de constater une erreur au niveau de l'exemple. En effet, dans la figure 5.1 de l'article, le lien entre H et G n'est pas transmis dans le graphe présent dans la colonne *reducer*. Pour le prouver il suffit d'ajouter le paramètre `debug` dans le constructeur de l'instance du graph utilisé de sorte à pouvoir afficher les différentes étapes de l'algorithme et notamment les graphes successifs obtenus après chaque itération. Remarquons à cet égard que les graphes successifs sont décrits par l'intermédiaire d'un dictionnaire.
Comme nous pouvons le constater à l'issue de la première itération il manque le lien entre H et G dans l'exemple de l'article.
Cependant le résultat final obtenu est bien identique à celui fourni dans l'article.

Pour rappel, le graphe d'exemple est donné ci-après :

<img src="figures/graphe_exemple.png" alt="Description de l'image" style="width:600px;">

In [1]:
class Graph:
    """
    Graphe exemple
    """

    def __init__(self, input_dict=None):
        self.graph = input_dict or {}

    def __str__(self):
        graph_sorted = {key: self.graph[key] for key in sorted(self.graph)}
        s = ""
        for key, values in graph_sorted.items():
            for value in values:
                s += f"{key} -> {value} \n"
        return s

    def __eq__(self, other):
        return self.graph == other.graph

    def ccf(self, debug=False):
        """
        Algorithme CCF
        """
        # Initialiser le nombre d'itérations
        iterations = 1
        previous_Graph = (
            self  # Initialisation pour stocker le graphe précédent
        )

        while True:
            # Effectuer les étapes mapper et reducer
            current_Graph = self.mapper(previous_Graph)
            current_Graph = self.reducer(current_Graph, debug)
            if debug:
                print(f"itération {iterations} : \n{current_Graph}")

            # Comparer le graphe précédent et le graphe actuel
            if previous_Graph == current_Graph:
                # Si les graphes sont identiques, arrêter la boucle
                if debug:
                    print(
                        f"Convergence atteinte après {iterations} itérations."
                    )
                return current_Graph, iterations

            # Mettre à jour le graphe précédent
            previous_Graph = current_Graph
            iterations += 1

    def mapper(self, input_Graph):
        """
        Créer un nouveau dictionnaire pour stocker le graphe bidirectionnel
        """
        bidirectional_graph = {}
        # Parcourir les nœuds du graphe original
        for node, neighbors in input_Graph.graph.items():
            # Ajouter chaque voisin à la liste des voisins du nœud courant
            if node not in bidirectional_graph:
                bidirectional_graph[node] = []
            for neighbor in neighbors:
                if neighbor not in bidirectional_graph:
                    bidirectional_graph[neighbor] = []
                # Ajouter les voisins dans les deux sens
                if neighbor not in bidirectional_graph[node]:
                    bidirectional_graph[node].append(neighbor)
                if node not in bidirectional_graph[neighbor]:
                    bidirectional_graph[neighbor].append(node)
        return Graph(bidirectional_graph)

    def map(self, key, value):
        """
        Fonction map
        """
        return f"""	emit({key},{value})
								emit({value},{key})
						"""

    def reduce(self, key, values, debug=False):
        # Initialiser la liste des valeurs et le compteur
        valueList = []
        CounterNewPair = 0
        min = key

        # Dictionnaire pour stocker les émissions
        emissions = {}

        # Trouver la valeur minimale et remplir valueList
        for value in values:
            if value < min:
                min = value
            valueList.append(value)

        # Vérifier si une émission est nécessaire
        if min < key:
            # Ajouter l'émission (key, min) au dictionnaire
            if key not in emissions:
                emissions[key] = []
            emissions[key].append(min)

            if debug:
                print(f"emit({key},{min})")

            # Ajouter les émissions pour les autres valeurs dans valueList
            for value in valueList:
                if min != value:
                    CounterNewPair += 1
                    if debug:
                        print(f"emit({value},{min})")

                    if value not in emissions:
                        emissions[value] = []
                    emissions[value].append(min)

        # Retourner ou enregistrer les émissions pour usage ultérieur
        return emissions

    def reducer(self, input_Graph, debug=False):
        emissions = []
        for key, value in input_Graph.graph.items():
            emissions.append(self.reduce(key, value, debug))
        return Graph(self.merge_dictionaries(emissions))

    def merge_dictionaries(self, dictionaries):
        # Dictionnaire final pour stocker les résultats
        final_dict = {}

        # Parcourir les dictionnaires successifs
        for current_dict in dictionaries:
            for key, values in current_dict.items():
                # Si la clef n'existe pas, initialiser une liste vide
                if key not in final_dict:
                    final_dict[key] = []
                # Ajouter les valeurs, en évitant les doublons
                for value in values:
                    if value not in final_dict[key]:
                        final_dict[key].append(value)
        return final_dict


# Exemple de graphe
input_dict = {
    "A": ["B"],
    "B": ["C", "D"],
    "D": ["E"],
    "F": ["G"],
    "G": ["H"],
}

debug = True
g = Graph(input_dict)
ccf_Graph, iterations = g.ccf(debug=debug)
print(
    f"A partir du graphe initial :\n{g}\net à l'issue de {iterations} itérations, le graphe suivant a été obtenu :\n{ccf_Graph}"
)

emit(B,A)
emit(C,A)
emit(D,A)
emit(C,B)
emit(D,B)
emit(E,B)
emit(E,D)
emit(G,F)
emit(H,F)
emit(H,G)
itération 1 : 
B -> A 
C -> A 
C -> B 
D -> A 
D -> B 
E -> B 
E -> D 
G -> F 
H -> F 
H -> G 

emit(B,A)
emit(C,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(B,A)
emit(D,A)
emit(B,A)
emit(E,A)
emit(E,B)
emit(D,B)
emit(G,F)
emit(H,F)
emit(H,F)
emit(G,F)
itération 2 : 
B -> A 
C -> A 
D -> A 
D -> B 
E -> A 
E -> B 
G -> F 
H -> F 

emit(B,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(D,A)
emit(B,A)
emit(E,A)
emit(B,A)
emit(G,F)
emit(H,F)
itération 3 : 
B -> A 
C -> A 
D -> A 
E -> A 
G -> F 
H -> F 

emit(B,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(G,F)
emit(H,F)
itération 4 : 
B -> A 
C -> A 
D -> A 
E -> A 
G -> F 
H -> F 

Convergence atteinte après 4 itérations.
A partir du graphe initial :
A -> B 
B -> C 
B -> D 
D -> E 
F -> G 
G -> H 

et à l'issue de 4 itérations, le graphe suivant a été obtenu :
B -> A 
C -> A 
D -> A 
E -> A 
G -> F 
H -> F 



## Implémentation de l'algorithme CCF en paradigme Spark

### Importation des librairies

In [ ]:
from abc import ABC, abstractmethod
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType

import pyspark
import time

### Création de la classe abstraite Graph

La représentation d'un graphe se fait par l'intermédiaire de la classe abstraite `Graph`. C'est à partir de celle-ci que deux autres classes, respectivement associées aux structures de type *Resilient Distributed Dataset* et *DataFrame*, seront créées.

La classe `Graph` est composée de trois attributs :
- `edges` : une liste d'éléments de type chaîne de caractères représentant les arêtes du graphe ;
- `numSlices` : un entier représentant le nombre de partitions du graphe (si `numSlices` n'est pas instancié, cette valeur est défini par `pyspark`);

La classe `Graph` est composée de plusieurs méthodes :
- `__init__` : le constructeur de la classe ;
- `__str__` : permet d'afficher le graphe ;
- `mapper` : permet de mapper les noeuds du graphe en créant un graphe bidirectionnel ;
- `reducer` : permet de réduire le graphe en fusionnant les noeuds connectés ;
- `ccf` : permet d'appliquer l'algorithme CCF sur le graphe ;
- `stop` : permet de stopper le contexte spark.

In [ ]:
class Graph(ABC):

  def __init__(self, edges, numSlices=None):
    self.spark_session = SparkSession \
      .builder \
      .appName("PySpark Hardcoded Graph") \
      .getOrCreate()
    self.spark_context = self.spark_session.sparkContext
    self.graph = None
    self.data_type = None

  @staticmethod
  def parallelize_edges(spark_context, edges, numSlices=None):
    """
    Convert a list of edges into a distributed RDD.

    Parameters:
      edges (list): List of graph edges as strings.
      numSlices (int): Number of partitions for the RDD.
      if None, use the default number of partitions defined by Spark.

    Returns:
      RDD: Distributed RDD containing the edges.
    """
    return spark_context.parallelize(edges, numSlices)

  @staticmethod
  def format_data(data):
    pass # méthode abstraite, doit être implémentée par les classes filles

  def __str__(self):
    return self.format_data(self.graph)

  @abstractmethod
  def mapper(self):
    pass # méthode abstraite, doit être implémentée par les classes filles

  @abstractmethod
  def reducer(self):
    pass # méthode abstraite, doit être implémentée par les classes filles

  def ccf(self, debug=False):
    nb_iteration = 0
    data = self.graph
    # Initialize accumulator here to be task-specific
    nb_new_pair = self.spark_context.accumulator(0)
    if debug:
      print(f"iteration {nb_iteration} \n{self}")
    while True:
      nb_iteration += 1
      start_pair = nb_new_pair.value
      # Perform mapping and reduction steps iteratively
      data = self.mapper(data)
      data_str_mapper = self.format_data(data)
      data = self.reducer(data, nb_new_pair)
      data_str_reducer = self.format_data(data)
      data = data.distinct()  # Remove duplicate edges to reduce redundancy
      data_str_reducer_distinct = self.format_data(data)
      if debug :
        print(f"iteration {nb_iteration} \nMapper: \n{data_str_mapper} \nReducer: \n{data_str_reducer} \nReducer (distinct) \n{data_str_reducer_distinct}")
        print(f"Iteration #{nb_iteration} - New pairs: {nb_new_pair.value}")
      if start_pair == nb_new_pair.value:
        if debug:
          print("\nNo new pairs, computation complete.")
        break
    return data

  @staticmethod
  def fetch(data):
    pass # méthode abstraite, doit être implémentée par les classes filles

  @staticmethod
  def get_connected_component_count(data):
    pass # méthode abstraite, doit être implémentée par les classes filles

  def workflow(self, debug=False):
    data = self.ccf(debug=debug)
    final_data_str = self.format_data(data)
    if debug:
      print(f"{self.data_type} obtenue à l'issue de l'algorithme CCF: \n{final_data_str}")
    # Extraction du résultat final (fetch)
    print("\nFinal Results (Node -> Component):")
    print(self.fetch(data))
    # Print the number of connected components
    print(f"\nNumber of connected components: {self.get_connected_component_count(data)}")

In [ ]:
class GraphRDD(Graph):
  def __init__(self, edges, numSlices=None):
    """
    Initialize the GraphRDD with a list of edges.
    """
    super().__init__(edges, numSlices)
    self.graph = self.parallelize_edges(self.spark_context, edges, numSlices).map(lambda x: x.split("\t")).map(lambda x: (x[0], x[1]))
    self.data_type = "RDD"	

  @staticmethod
  def format_data(data):
    sorted_edges = sorted(data.collect(), key=lambda x: (x[0], x[1]))
    formatted_edges = [f"{source} -> {destination}" for source, destination in sorted_edges]
    formatted_edges = "\n".join(formatted_edges)
    return formatted_edges

  @staticmethod
  def mapper(rdd):
    return rdd.union(rdd.map(lambda x: (x[1], x[0])))

  def reducer(self, data, nb_new_pair):
    """
    Perform the reduce step of the algorithm and propagate the smallest component ID for Resilient Distributed Dataset
    """
    def count_nb_new_pair(x, accumulator):
        """
        Count new pairs and propagate smallest ID
        """
        k, values = x
        min_id, value_list = k, []
        for v in values:
            if v < min_id:
                min_id = v
            value_list.append(v)
        if min_id < k:
            yield(k, min_id)
            for v in value_list:
                if v != min_id:
                    accumulator.add(1)
                    yield(v, min_id)
    # Group edges by source node and propagate smallest component ID
    return data.groupByKey().flatMap(lambda x: count_nb_new_pair(x, nb_new_pair)).sortByKey()

  @staticmethod
  def fetch(data):
    output_str = ""
    data = data.map(lambda x: (x[0], x[1])).distinct().collect()
    for node, component in sorted(data):
      output_str += f"Node {node} belongs to Component {component}\n"
    return output_str

  @staticmethod
  def get_connected_component_count(data):
    return data.map(lambda x: (x[0], x[1])).distinct().count()

In [ ]:
class GraphDF(Graph):
  def __init__(self, edges, numSlices=None):
    """
    Initialize the GraphDF with a list of edges.
    """
    super().__init__(edges, numSlices)
    # The list of edges is parallelized into an RDD.
    # The RDD is then converted into a DataFrame with a single column "raw" using createDataFrame.
    df = self.spark_session.createDataFrame(self.parallelize_edges(self.spark_context, edges, numSlices).map(lambda x: (x,)), ["raw"])
    self.graph = df.withColumn('k', split(df['raw'], '\t').getItem(0)) \
                  .withColumn('v', split(df['raw'], '\t').getItem(1)) \
                  .drop('raw')
    self.data_type = "DataFrame"

  @staticmethod
  def format_data(df):
    rows = df.selectExpr("k as source", "v as destination").collect()
    sorted_edges = sorted(rows, key=lambda x: (x[0], x[1]))
    formatted_edges = [f"{row['source']} -> {row['destination']}" for row in sorted_edges]
    # Join all formatted edges with newlines
    return "\n".join(formatted_edges)

  @staticmethod
  def mapper(data):
    return data.union(data.select(col("v").alias("k"), col("k").alias("v")))

  def reducer(self, data, nb_new_pair):
    """
    Perform the reduce step of the algorithm and propagate the smallest component ID for DateFrame
    """
    # Group by source node, propagate smallest component ID, and count new pairs
    data = data.groupBy(col("k")).agg(collect_set("v").alias("v"))\
        .withColumn("min", least(col("k"), array_min("v")))\
        .filter((col("k") != col("min")))

    # Count the number of new pairs added in this iteration
    nb_new_pair += data.withColumn("count", size("v") - 1).select(sum("count")).collect()[0][0]

    data = data.select(
      col("min").alias("a_min"),
      expr("filter(concat(array(k), v), x -> x != min)").alias("valueList")
    )
    data = data.select(
      explode(col("valueList")).alias("k"),  # Décomposer `valueList` en une ligne par élément
      col("a_min").alias("v")  # Conserver `a_min`
    )
    return data

  @staticmethod
  def fetch(data):
    output_str = ""
    data = data.select("k", "v").distinct().collect()
    for row in sorted(data, key=lambda x: x["k"]):
      output_str += f"Node {row['k']} belongs to Component {row['v']}\n"
    return output_str

  @staticmethod
  def get_connected_component_count(data):
    return data.select('k').distinct().count()

In [ ]:
if __name__ == "__main__":
  # List of edges in the graph
  edges = [
       "A\tB",
       "B\tC",
       "B\tD",
       "D\tE",
       "F\tG",
       "G\tH"
  ]
  numSlices = None
  debug = True
  # Create GraphRDD and GraphDF instances
  RDD = GraphRDD(edges, numSlices=numSlices)
  DF = GraphDF(edges, numSlices=numSlices)
  # Execute CCF algorithm on RDD instance
  RDD.workflow(debug)
  # Execute CCF algorithm on DF instance
  DF.workflow(debug)